# Local RAG Evaluation

Objective : Evaluate a local lightweight LLM setup for NLP tasks while ensuring zero cost and full data privacy.
It leverages the Qwen 2.5 0.5B model for fast, CPU-based inference alongside DistilRoBERTa embeddings.
Precomputed document embeddings are used to represent textual data efficiently.
The workflow focuses on clustering or structuring text data based on these embeddings.
Overall, the objective is to demonstrate an efficient, fully local pipeline for text representation and clustering using small models.

By running a language model directly on local hardware, we ensure zero-cost querying and maintain complete data privacy.

* **The Model:** `Qwen 2.5 0.5B`. This is an ultra-lightweight, 0.5-billion (500 million) parameter model. Small-scale models like this are specifically optimized to deliver surprisingly robust performance for basic NLP tasks while remaining capable of running extremely fast on standard consumer CPUs, entirely avoiding the need for dedicated graphics cards (GPUs) or high-end servers.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath(os.path.join("..")))
from src.news_clustering import parse_embedding_string

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import torch
from transformers import pipeline

### Loading the models Qwen and distilRoBERTa

In [14]:
# ============================================================
# 1. Loading models
# ============================================================
print("Loading DistilRoBERTa (Retrieval)...")
embedder = SentenceTransformer("sentence-transformers/all-distilroberta-v1", device="cpu")

print("Loading Qwen 0.5B LLM (Generation)...")
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct", 
    device="cpu", 
    dtype=torch.float32 
)

Loading DistilRoBERTa (Retrieval)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4098.37it/s]
RobertaModel LOAD REPORT from: sentence-transformers/all-distilroberta-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Qwen 0.5B LLM (Generation)...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 329.36it/s]


In [18]:
# --- DistilRoBERTa ---
print("\n EMBEDDING MODEL (DistilRoBERTa):")
# The embedder object contains several modules, the actual model is the first one [0]
roberta_config = embedder[0].auto_model.config

print(f"- Generated vector dimension: {embedder.get_sentence_embedding_dimension()}")
print(f"- Max text length (Tokens): {embedder.max_seq_length}")
print(f"- Vocabulary size: {roberta_config.vocab_size:,}")
print(f"- Base architecture: {roberta_config.architectures[0]}")


 EMBEDDING MODEL (DistilRoBERTa):
- Generated vector dimension: 768
- Max text length (Tokens): 512
- Vocabulary size: 50,265
- Base architecture: RobertaForMaskedLM


In [20]:
# --- Qwen 0.5B ---
print("\n GENERATION MODEL (Qwen 0.5B):")
# Accessing the underlying model via the pipeline
qwen_model = llm.model
qwen_config = qwen_model.config

# Formatting the number of parameters with spaces for readability
num_params = qwen_model.num_parameters()
print(f"- Total number of parameters: {num_params:,}".replace(',', ' '))
print(f"- Maximum context size: {qwen_config.max_position_embeddings:,}".replace(',', ' '))
print(f"- Vocabulary size: {qwen_config.vocab_size:,}".replace(',', ' '))
print(f"- Precision (Dtype): {qwen_model.dtype}")
print(f"- Base architecture: {qwen_config.architectures[0]}")


 GENERATION MODEL (Qwen 0.5B):
- Total number of parameters: 494 032 768
- Maximum context size: 32 768
- Vocabulary size: 151 936
- Precision (Dtype): torch.float32
- Base architecture: Qwen2ForCausalLM


### Loading precomputed document embeddings by distilroberta

In [21]:
# Loading precomputed document embeddings
df_news = pd.read_csv("../data/for_models/news_features_bpe.csv")
document_embeddings = np.stack(df_news["embedding"].apply(parse_embedding_string).values)

In [22]:
document_embeddings.shape

(736, 768)

### RAG Pipeline

In [23]:
# ============================================================
# 2. RETRIEVAL ENGINE 
# ============================================================
def retrieve_relevant_articles(question, top_k=5):
    question_vector = embedder.encode([question])
    similarities = cosine_similarity(question_vector, document_embeddings)[0] 
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    retrieved_docs = []
    for idx in top_indices:
        row = df_news.iloc[idx]
        score_pct = similarities[idx] * 100 
        doc_text = f"- [Relevance: {score_pct:.1f}%] [{row['date']}] {row['headline']}"
        retrieved_docs.append(doc_text)
        
    return "\n".join(retrieved_docs), similarities[top_indices]

In [29]:
# ============================================================
# 3. GENERATION (Adapted for Hugging Face)
# ============================================================
def ask_database_hf(question, top_k=5):
    print(f"\n Searching for the top {top_k} relevant articles...")
    context, scores = retrieve_relevant_articles(question, top_k=top_k)
    
    print(" Querying the LLM (Hugging Face)...")
    
    # Specific formatting for Qwen
    messages = [
        {"role": "system", "content": "You are an expert financial analyst. Answer ONLY using the provided context. If the answer is not in the context, clearly state it."},
        {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}
    ]
    
    prompt = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    outputs = llm(
        prompt, 
        max_new_tokens=200, 
        temperature=0.1, 
        do_sample=True,
        pad_token_id=llm.tokenizer.eos_token_id
    )
    
    # Extract the response
    answer = outputs[0]["generated_text"].split("<|im_start|>assistant\n")[-1].strip()
    
    print("\n================ ANSWER ================\n")
    print(answer)
    print("\n================ SOURCES USED ================")
    print(context)

### Question

In [ ]:
# 1. Ask your question
ma_question = "What is happening in Iran?"

# 2. Call the function (e.g., requesting the top 5 most relevant articles)
ask_database_hf(question=ma_question, top_k=5)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Searching for the top 5 relevant articles...
 Querying the LLM (Hugging Face)...

================ ANSWER ================

Iran is currently experiencing a conflict with its neighbor, the United States, which has led to tensions and potential diplomatic issues. The country is also facing challenges related to its energy supply, particularly from the Strait of Hormuz, due to disruptions caused by Iran's threat to withdraw its oil exports if the US imposes sanctions on it. Additionally, there have been concerns about the economic impact of these geopolitical events on both countries.

================ SOURCES USED ================
- [Relevance: 23.8%] [2026-03-02] Iran conflict: Where things stand, global responses — and what comes next
- [Relevance: 20.4%] [2026-02-28] 3 themes that drove Wall Street's wild week and the new U.S.-Iran conflict wildcard
- [Relevance: 19.6%] [2026-03-23] Iran threatens U.S. Treasury buyers as Trump’s 48-hour ultimatum looms
- [Relevance: 17.4%] [2026-03